In [ ]:
import pickle
import numpy as np

from src.linear_correlations import CorrelationGenerator
from src.resnet.model_with_embedding import FiniteResNetWithEmbedding
from src.resnet.activations import linear, linear_der

MAIN_SEED = 42
np.random.seed(MAIN_SEED)

In [ ]:
d_in = 3
d_out = 3

N = 1

x_scale = 0.8
x_true = np.random.randn(d_in)
x_true = x_scale*(x_true/np.linalg.norm(x_true))

A_true = np.random.randn(d_out, d_in)
noise_level = 0.05
y_true = A_true @ x_true + noise_level * np.random.randn(d_out)

In [ ]:
x_true, y_true, A_true

Training Parameters

In [ ]:
K = 15
eta_0 = 0.1

Infinite Model

In [ ]:
# Given how the model's written, essentially all sigmas are 1
infinite_model = CorrelationGenerator(
    x=x_true,
    y=y_true,
    sigma_u=1, 
    sigma_v=1, 
    a=1, 
    eta_u=eta_0, 
    eta_v=eta_0, 
    sigma_we=1, 
    sigma_wu=1,
    K=K,
    num_s=100
    )

In [ ]:
result = infinite_model.run()

In [ ]:
def train_track(model, X, Y, K, eta, eta_u, eta_v):
    losses = []
    H_track = []
    B_track = []
    out_track = []
    
    for k in range(K):
        loss, H, B, out = model.step(X, Y, eta, eta_u, eta_v, track=True)
        losses.append(loss)
        
        H_track.append(H.copy())
        B_track.append(B.copy())
        out_track.append(out.copy())
        print(f"[{k+1}/{K}] loss = {loss:.5f}")
        
    return np.array(losses), np.array(H_track), np.array(B_track), np.array(out_track)

Create single large W_e, W_u for coupled training run.

In [ ]:
max_D = 1000

W_SEED = 4096 # Modify this to obtain results for different random initializations.
np.random.seed(W_SEED)
rootW_e = np.random.randn(max_D, d_in)
rootW_u = np.random.randn(max_D, d_out)

In [ ]:
with open(f"data/exp_track_rate/W_e_maxD{max_D}_din{d_in}_MAIN_SEED{MAIN_SEED}_WSEED{W_SEED}.pkl", "wb") as f:
    pickle.dump(rootW_e, f)
with open(f"data/exp_track_rate/W_u_maxD{max_D}_dout{d_out}_MAIN_SEED{MAIN_SEED}_WSEED{W_SEED}.pkl", "wb") as f:
    pickle.dump(rootW_u, f)

Simulate explicit trajectories of $H$ and $B$

In [ ]:
from src.linear_simulation import simulate_system_batched_optimized

H_ds_raw, B_ds_raw = simulate_system_batched_optimized(
    rootW_e, rootW_u,
    infinite_model.Gamma_H, infinite_model.Gamma_B,
    infinite_model.A, infinite_model.A_tilde,
    infinite_model.grad_loss_array, infinite_model.x,
)

In [ ]:
from scipy.interpolate import interp1d
def interpolate(H_d, new_len):
    x_old = np.linspace(0, 1, H_d.shape[0])
    x_new = np.linspace(0, 1, new_len)
    f = interp1d(x_old, H_d, axis=0, kind='linear')
    return f(x_new)  # shape (501, 15)

## Experiments

In [ ]:
# ── sizes ──────────────────────────────────────
n_L = 15
n_M = 15
n_D = 15
range_L = 500
range_M = 500
range_D = 500
N_reps = 5

def logspace_int(lo, hi, n):
    """Log-spaced integers (floor), matching Julia's convention."""
    return np.floor(10**(np.linspace(np.log10(lo), np.log10(hi), n))).astype(int)

Ls = logspace_int(10, range_L, n_L)
Ms = logspace_int(10, range_M, n_M)
Ds = logspace_int(10, range_D, n_D)

print("Ls:", Ls)
print("Ms:", Ms)
print("Ds:", Ds)

In [ ]:
import time

# ── tracking arrays ────────────────────────────
outputs_track = np.zeros((N_reps, n_L, n_M, n_D, K, d_out))
loss_track = np.zeros((N_reps, n_L, n_M, n_D, K))
mean_hs_track = np.zeros((N_reps, n_L, n_M, n_D, K)) # D L+1, K
mean_bs_track = np.zeros((N_reps, n_L, n_M, n_D, K)) # D L+1, K
max_hs_track = np.zeros((N_reps, n_L, n_M, n_D, K)) # D L+1, K
max_bs_track = np.zeros((N_reps, n_L, n_M, n_D, K)) # D L+1, K

loop_SEED = 0

# ── main loop ──────────────────────────────────
t0 = time.time()

for i in range(n_D):       
    for ii in range(n_L):
        for iii in range(n_M - 1):
            print(f"i={i+1}, ii={ii+1}, iii={iii+1}")
            D = int(Ds[i])
            L = int(Ls[ii])
            M = int(Ms[iii])

            for rep in range(N_reps):
                rep_seed = loop_SEED + rep
                model = FiniteResNetWithEmbedding(D, M, L, d_in = d_in, d_out=d_out, alpha=1.0, activation=linear, activation_der=linear_der, seed=rep_seed)
                model.W_e = rootW_e[:D]
                model.W_u = rootW_u[:D]
                scale_eta_v, scale_eta_u = D, D
                loss, Hs, Bs, model_outputs = train_track(model, x_true.reshape(-1,1), y_true.reshape(-1,1), K, eta_0, scale_eta_u, scale_eta_v)
                
                outputs_track[rep, ii, iii, i] = model_outputs[:,:,0]
                loss_track[rep, ii, iii, i] = loss

                H_ds = np.stack([interpolate(H_ds_raw[d], L+1) for d in range(D)])
                B_ds = np.stack([interpolate(B_ds_raw[d], L+1) for d in range(D)])
                h_ds = Hs[:, :, 0, :].transpose(1,2,0)
                b_ds = Bs[:, :, 0, :].transpose(1,2,0)
                norm_bar_h = np.mean(np.abs(h_ds - H_ds), axis=0)
                norm_bar_b = np.mean(np.abs(b_ds - B_ds), axis=0)

                mean_hs_track[rep, ii, iii, i] = np.mean(norm_bar_h, axis=0)
                max_hs_track[rep, ii, iii, i] = np.max(norm_bar_h, axis=0)
                mean_bs_track[rep, ii, iii, i] = np.mean(norm_bar_b, axis=0)
                max_bs_track[rep, ii, iii, i] = np.max(norm_bar_b, axis=0)

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.2f}s")

In [ ]:
with open(f"data/exp_track_rate/outputs_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f: 
    pickle.dump(outputs_track, f)
with open(f"data/exp_track_rate/loss_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:  
    pickle.dump(loss_track, f)
with open(f"data/exp_track_rate/mean_hs_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:  
    pickle.dump(mean_hs_track, f)
with open(f"data/exp_track_rate/max_hs_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:   
    pickle.dump(max_hs_track, f)
with open(f"data/exp_track_rate/mean_bs_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:  
    pickle.dump(mean_bs_track, f)
with open(f"data/exp_track_rate/max_bs_track_D{range_D}_L{range_L}_M{range_M}_reps{N_reps}_K{K}_loop_SEED{loop_SEED}_WSEED{W_SEED}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:   
    pickle.dump(max_bs_track, f)